# Baseline Evaluation — Llama 3.2 3B Zero-shot + Few-shot

**Goal:** Establish baseline metrics untuk Indonesian customer support intent classification sebelum fine-tuning.

**Setup:**
1. Runtime → Change runtime type → GPU (T4 free tier OK)
2. Upload `data/splits/test.jsonl` + `data/intents.json` (atau clone repo)
3. Login ke HuggingFace (butuh access ke `meta-llama/Llama-3.2-3B-Instruct`, request di HF hub page)

**Runtime:** ~10-15 menit di T4.

## 1. Install dependencies

In [ ]:
!pip install -q transformers>=4.45.0 accelerate>=1.0.0 scikit-learn seaborn tqdm

## 2. Clone repo (atau upload files manual)

In [ ]:
# Kalau repo public:
# !git clone https://github.com/aulia/fine-tuning-lab
# %cd fine-tuning-lab

# Kalau upload manual, verify files ada:
import os
assert os.path.exists('data/intents.json'), 'Upload data/intents.json'
assert os.path.exists('data/splits/test.jsonl'), 'Upload data/splits/test.jsonl'
assert os.path.exists('eval/metrics.py'), 'Upload eval/metrics.py'

## 3. Login ke HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Load model + tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'Model loaded: {MODEL_ID}')
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 5. Load data + intents

In [ ]:
import json
from pathlib import Path

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

test_samples = load_jsonl('data/splits/test.jsonl')
intents_data = json.loads(Path('data/intents.json').read_text())
intents = intents_data['intents']
labels = [i['label'] for i in intents]
label_desc = {i['label']: i['description_id'] for i in intents}

print(f'Test set: {len(test_samples)} samples')
print(f'Labels ({len(labels)}):', labels)

## 6. Zero-shot prompt template

In [ ]:
def build_zeroshot_prompt(text, labels, label_desc):
    intent_list = '\n'.join(f'{i+1}. {label} — {label_desc[label]}' for i, label in enumerate(labels))
    return [
        {'role': 'system', 'content': (
            f'Kamu adalah intent classifier untuk customer support Indonesian e-commerce. '
            f'Klasifikasi pesan user ke salah satu dari {len(labels)} intent berikut:\n\n{intent_list}\n\n'
            f'Output HANYA nama label (contoh: "check_order_status"), tanpa penjelasan.'
        )},
        {'role': 'user', 'content': f'Message: {text}\n\nIntent:'},
    ]

# Preview
sample = test_samples[0]
prompt = build_zeroshot_prompt(sample['text'], labels, label_desc)
print(f'Sample text: {sample["text"]}')
print(f'True label:  {sample["label"]}')
print(f'\nSystem prompt length: {len(prompt[0]["content"])} chars')

## 7. Inference function

In [ ]:
import re

def predict(messages, max_new_tokens=15):
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy for determinism
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    return generated.strip()

def extract_label(raw, labels):
    """Extract label dari raw model output. Fallback ke general_greeting kalau nggak match."""
    raw_lower = raw.lower()
    # Exact match check
    for label in labels:
        if label.lower() in raw_lower:
            return label
    # Fallback
    return 'general_greeting'

# Smoke test
raw = predict(build_zeroshot_prompt(sample['text'], labels, label_desc))
print(f'Raw output: {raw!r}')
print(f'Extracted:  {extract_label(raw, labels)}')
print(f'True:       {sample["label"]}')

## 8. Run zero-shot eval

In [ ]:
from tqdm.auto import tqdm

zeroshot_preds = []
for sample in tqdm(test_samples, desc='Zero-shot'):
    prompt = build_zeroshot_prompt(sample['text'], labels, label_desc)
    raw = predict(prompt)
    pred = extract_label(raw, labels)
    zeroshot_preds.append(pred)

y_true = [s['label'] for s in test_samples]
print('\n✓ Zero-shot inference complete')

## 9. Zero-shot metrics

In [ ]:
import sys
sys.path.insert(0, '.')
from eval.metrics import evaluate, print_report, save_metrics, plot_confusion

zs_metrics = evaluate(y_true, zeroshot_preds, labels)
print_report(zs_metrics, 'Baseline: Zero-shot Llama 3.2 3B Instruct')
save_metrics(zs_metrics, 'results/baseline_zeroshot.json')
plot_confusion(zs_metrics, save_path='results/baseline_zeroshot_cm.png');

## 10. Few-shot prompt template (3 examples per intent)

In [ ]:
# Sample 3 examples per intent dari TRAIN split (bukan test — no leakage)
train_samples = load_jsonl('data/splits/train.jsonl')

import random
random.seed(42)

examples_by_label = {}
for s in train_samples:
    examples_by_label.setdefault(s['label'], []).append(s['text'])

fewshot_examples = {}
for label, texts in examples_by_label.items():
    random.shuffle(texts)
    fewshot_examples[label] = texts[:3]  # 3 examples per intent

def build_fewshot_prompt(text, labels, label_desc, examples):
    intent_list = '\n'.join(f'{i+1}. {label} — {label_desc[label]}' for i, label in enumerate(labels))

    examples_text = ''
    for label in labels:
        for ex in examples[label]:
            examples_text += f'\nMessage: {ex}\nIntent: {label}\n'

    return [
        {'role': 'system', 'content': (
            f'Kamu adalah intent classifier. Klasifikasi pesan user ke salah satu dari {len(labels)} intent:\n\n{intent_list}\n\n'
            f'Berikut contoh untuk tiap intent:{examples_text}\n'
            f'Output HANYA nama label, tanpa penjelasan.'
        )},
        {'role': 'user', 'content': f'Message: {text}\n\nIntent:'},
    ]

# Preview length
prompt = build_fewshot_prompt(sample['text'], labels, label_desc, fewshot_examples)
print(f'System prompt length: {len(prompt[0]["content"])} chars')
print(f'Token estimate: ~{len(prompt[0]["content"]) // 4} token')

## 11. Run few-shot eval

In [ ]:
fewshot_preds = []
for sample in tqdm(test_samples, desc='Few-shot'):
    prompt = build_fewshot_prompt(sample['text'], labels, label_desc, fewshot_examples)
    raw = predict(prompt)
    pred = extract_label(raw, labels)
    fewshot_preds.append(pred)

print('\n✓ Few-shot inference complete')

## 12. Few-shot metrics + comparison

In [ ]:
fs_metrics = evaluate(y_true, fewshot_preds, labels)
print_report(fs_metrics, 'Baseline: Few-shot (3 examples per intent) Llama 3.2 3B')
save_metrics(fs_metrics, 'results/baseline_fewshot.json')
plot_confusion(fs_metrics, save_path='results/baseline_fewshot_cm.png');

from eval.metrics import compare_metrics
compare_metrics(zs_metrics, fs_metrics, 'Zero-shot vs Few-shot')

## 13. Error analysis — sample misclassifications

Look at 10 samples yang few-shot mis-classify. Cari pattern buat inform data augmentation atau fine-tune strategy.

In [ ]:
wrong = [(s, p) for s, p in zip(test_samples, fewshot_preds) if s['label'] != p]
print(f'Total mis-classifications: {len(wrong)}/{len(test_samples)}')

print('\nSample misclassifications:\n')
for s, pred in wrong[:10]:
    print(f'  Text:  {s["text"]}')
    print(f'  True:  {s["label"]}')
    print(f'  Pred:  {pred}')
    print()

## 14. Save final baseline summary

Format: markdown ready-to-paste ke README atau blog post.

In [ ]:
summary = f'''# Baseline Evaluation Summary

**Model:** meta-llama/Llama-3.2-3B-Instruct
**Test set:** {len(test_samples)} samples, {len(labels)} classes

## Results

| Method | Accuracy | Macro F1 | Weighted F1 |
|---|---|---|---|
| Zero-shot | {zs_metrics["accuracy"]:.4f} | {zs_metrics["macro_f1"]:.4f} | {zs_metrics["weighted_f1"]:.4f} |
| Few-shot (3 examples per intent) | {fs_metrics["accuracy"]:.4f} | {fs_metrics["macro_f1"]:.4f} | {fs_metrics["weighted_f1"]:.4f} |

**Bootstrap 95% CI accuracy:**
- Zero-shot: [{zs_metrics["accuracy_ci_95"]["lower"]:.4f}, {zs_metrics["accuracy_ci_95"]["upper"]:.4f}]
- Few-shot:  [{fs_metrics["accuracy_ci_95"]["lower"]:.4f}, {fs_metrics["accuracy_ci_95"]["upper"]:.4f}]

## Next: Minggu 7 fine-tuning
- Target: accuracy 90%+, macro F1 88%+ pakai LoRA r=8 alpha=16 lr=2e-4 3-5 epochs
- Improvement target: minimum +20 pp dari best baseline (few-shot)
'''

with open('results/baseline_summary.md', 'w') as f:
    f.write(summary)

print(summary)